In [ ]:
# source: https://www.kaggle.com/discussions/general/74235
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

# Needed for Lemmatization
! python3 -m spacy download en_core_web_sm

# MBTI Datasets
# ! kaggle datasets download -d "zeyadkhalid/mbti-personality-types-500-dataset"
! kaggle datasets download -d "mazlumi/mbti-personality-type-twitter-dataset"
#! kaggle datasets download -d "datasnaek/mbti-type" # included in first one

! unzip "*.zip"

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dataset URL: https://www.kaggle.com/datasets/mazlumi/mbti-personality-type-twitter-dataset
License(s): other
mbti-personality-type-twitter-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/datasnaek/mbti-type
License(s): CC0-1.0
mbti-type.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  mbti-personality-type-twitter-dataset.zip
replace twitter_MBTI.csv? [y]es, [n]o, [A]ll, [N]one

In [ ]:
import pandas as pd
import numpy as np

# Matplotlib, plt and cm modules
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Seaborn for plotting
import seaborn as sns

# Pritty print
from pprint import pprint

# for reading .env vars
import os

# For cleaning posts
import re

# For lemmanization
import spacy

DATASETS_PATH = './'

- Make labels Uppercase (intj->INTJ)
- No punctuations, stopwords, URLs
- Lemmatization (Running->Run, improves NLP)
- Reconstruct samples to be equal-sized chunks (500 words per sample)

In [ ]:
data = pd.read_csv(f'{DATASETS_PATH}twitter_MBTI.csv')
data

,Unnamed: 0,text,label
0,0,@Pericles216 @HierBeforeTheAC @Sachinettiyil T...,intj
1,1,@Hispanthicckk Being you makes you look cute||...,intj
2,2,@Alshymi Les balles sont réelles et sont tirée...,intj
3,3,"I'm like entp but idiotic|||Hey boy, do you wa...",intj
4,4,@kaeshurr1 Give it to @ZargarShanif ... He has...,intj
...,...,...,...
7806,7806,"@sobsjjun God,,pls take care 😕|||@sobsjjun Hir...",intp
7807,7807,@Ignis_02 wow last time i got intp https://t.c...,intp
7808,7808,@akupilled A 100%|||@akupilled That SOMEONE wi...,entp
7809,7809,If you’re #INTJ this one is for you | What is ...,infj


In [ ]:
data.drop(columns=['Unnamed: 0'], inplace=True)

,text,label
0,@Pericles216 @HierBeforeTheAC @Sachinettiyil T...,intj
1,@Hispanthicckk Being you makes you look cute||...,intj
2,@Alshymi Les balles sont réelles et sont tirée...,intj
3,"I'm like entp but idiotic|||Hey boy, do you wa...",intj
4,@kaeshurr1 Give it to @ZargarShanif ... He has...,intj
...,...,...
7806,"@sobsjjun God,,pls take care 😕|||@sobsjjun Hir...",intp
7807,@Ignis_02 wow last time i got intp https://t.c...,intp
7808,@akupilled A 100%|||@akupilled That SOMEONE wi...,entp
7809,If you’re #INTJ this one is for you | What is ...,infj


In [ ]:
# Make labels uppercase to match the other dataset
data['label'] = data['label'].str.upper()
data

,text,label
0,@Pericles216 @HierBeforeTheAC @Sachinettiyil T...,INTJ
1,@Hispanthicckk Being you makes you look cute||...,INTJ
2,@Alshymi Les balles sont réelles et sont tirée...,INTJ
3,"I'm like entp but idiotic|||Hey boy, do you wa...",INTJ
4,@kaeshurr1 Give it to @ZargarShanif ... He has...,INTJ
...,...,...
7806,"@sobsjjun God,,pls take care 😕|||@sobsjjun Hir...",INTP
7807,@Ignis_02 wow last time i got intp https://t.c...,INTP
7808,@akupilled A 100%|||@akupilled That SOMEONE wi...,ENTP
7809,If you’re #INTJ this one is for you | What is ...,INFJ


In [17]:
# source: https://www.kaggle.com/code/hadia150/advancedmbti-textclassification
def clean_post(text):
    """
    Clean social media text by removing noise
    """
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove usernames (@user) ----> Written by Jhonatan Parada
    text = re.sub(r'@\w+\b','',text)

    # Remove post separator
    text = text.replace('|||', ' ')

    # Remove special characters but keep spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [18]:
data['clean_posts'] = data['text'].apply(clean_post)

print(data.isnull().sum())

data

text           0
label          0
clean_posts    0
dtype: int64


,text,label,clean_posts
0,@Pericles216 @HierBeforeTheAC @Sachinettiyil T...,INTJ,the pope is infallible this is a catholic dogm...
1,@Hispanthicckk Being you makes you look cute||...,INTJ,being you makes you look cute on because then ...
2,@Alshymi Les balles sont réelles et sont tirée...,INTJ,les balles sont relles et sont tires trs rapid...
3,"I'm like entp but idiotic|||Hey boy, do you wa...",INTJ,im like entp but idiotic hey boy do you want t...
4,@kaeshurr1 Give it to @ZargarShanif ... He has...,INTJ,give it to he has pica since childhood say qub...
...,...,...,...
7806,"@sobsjjun God,,pls take care 😕|||@sobsjjun Hir...",INTP,godpls take care hiro emergency room are you o...
7807,@Ignis_02 wow last time i got intp https://t.c...,INTP,wow last time i got intp i think u upset the f...
7808,@akupilled A 100%|||@akupilled That SOMEONE wi...,ENTP,a that someone will get his ass kicked so its ...
7809,If you’re #INTJ this one is for you | What is ...,INFJ,if youre intj this one is for you what is neve...


In [22]:
# Count words and make them 500 after cleaning

# There are too many words per entry, let's truncate them to 500 and then apply
# the lemmatizer function to save time.
data['clean_posts'].apply(lambda row: len(row.split()))

,clean_posts
0,2036
1,888
2,1571
3,1115
4,832
...,...
7806,758
7807,953
7808,1080
7809,1363


In [23]:
max_words = 500
data['clean_posts'] = data['clean_posts'].apply(lambda x: ' '.join(str(x).split()[:max_words]))

data['clean_posts'].apply(lambda row: len(row.split()))

,clean_posts
0,500
1,500
2,500
3,500
4,500
...,...
7806,500
7807,500
7808,500
7809,500


In [38]:
# source: https://www.kaggle.com/code/rajshreev/mbti-personality-predictor-using-machine-learning
def get_types(row):
    t=row['label'] # or t

#    I = 0; N = 0
#    T = 0; J = 0

    if t[0] == 'I': I = 'I'
    elif t[0] == 'E': I = 'E'
    else: print('I-E not found')

    if t[1] == 'N': N = 'N'
    elif t[1] == 'S': N = 'S'
    else: print('N-S not found')

    if t[2] == 'T': T = 'T'
    elif t[2] == 'F': T = 'F'
    else: print('T-F not found')

    if t[3] == 'J': J = 'J'
    elif t[3] == 'P': J = 'P'
    else: print('J-P not found')
    return pd.Series( {'ie':I, 'ns':N , 'tf': T, 'jp': J })

data = data.join(data.apply (lambda row: get_types (row),axis=1))
data.head(5)



KeyError: 'labels'

In [26]:
# source: https://www.geeksforgeeks.org/machine-learning/python-pos-tagging-and-lemmatization-using-spacy/
nlp = spacy.load('en_core_web_sm')  # python -m spacy download en_core_web_sm

def lemmatize_text(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

# ONLY RUN WHEN READY TO SAVE THE FILE
data['lemmatize_clean_posts'] = data['clean_posts'].apply(lemmatize_text)

# THIS IS ONLY FOR 3 EXAMPLE
# data['clean_posts'].head(3).apply(lemmatize_text)

# Task: when cleaning and lemmatizing, download data to csv because it takes a while to run.

In [46]:
data.drop(columns=['text','clean_posts'], inplace=True)
data


,label,ie,ns,tf,jp,lemmatize_clean_posts
0,INTJ,I,N,T,J,the pope be infallible this be a catholic dogm...
1,INTJ,I,N,T,J,be you make you look cute on because then I ca...
2,INTJ,I,N,T,J,les balle sont relle et sont tire tr rapidemen...
3,INTJ,I,N,T,J,I m like entp but idiotic hey boy do you want ...
4,INTJ,I,N,T,J,give it to he have pica since childhood say qu...
...,...,...,...,...,...,...
7806,INTP,I,N,T,P,godpls take care hiro emergency room be you ok...
7807,INTP,I,N,T,P,wow last time I get intp I think u upset the f...
7808,ENTP,E,N,T,P,a that someone will get his ass kick so its ok...
7809,INFJ,I,N,F,J,if you re intj this one be for you what be nev...


In [51]:
# To reduce the dataset filesize, let's drop text, and clean_posts

data.loc[:,['lemmatize_clean_posts', 'label', 'ie', 'ns', 'tf', 'jp']]

,lemmatize_clean_posts,label,ie,ns,tf,jp
0,the pope be infallible this be a catholic dogm...,INTJ,I,N,T,J
1,be you make you look cute on because then I ca...,INTJ,I,N,T,J
2,les balle sont relle et sont tire tr rapidemen...,INTJ,I,N,T,J
3,I m like entp but idiotic hey boy do you want ...,INTJ,I,N,T,J
4,give it to he have pica since childhood say qu...,INTJ,I,N,T,J
...,...,...,...,...,...,...
7806,godpls take care hiro emergency room be you ok...,INTP,I,N,T,P
7807,wow last time I get intp I think u upset the f...,INTP,I,N,T,P
7808,a that someone will get his ass kick so its ok...,ENTP,E,N,T,P
7809,if you re intj this one be for you what be nev...,INFJ,I,N,F,J


In [52]:
data.to_csv("cleaned_twitter_MBTI.csv", index=False)

In [ ]:
# Concatenate or perform union for both datasets

In [31]:
! kaggle datasets download -d "zeyadkhalid/mbti-personality-types-500-dataset"
! unzip "mbti-personality-types-500-dataset.zip"

Dataset URL: https://www.kaggle.com/datasets/zeyadkhalid/mbti-personality-types-500-dataset
License(s): CC0-1.0
100% 123M/123M [00:01<00:00, 99.2MB/s]

Archive:  mbti-personality-types-500-dataset.zip
  inflating: MBTI 500.csv            


In [34]:
data2 = pd.read_csv("MBTI 500.csv")
data2

,posts,type
0,know intj tool use interaction people excuse a...,INTJ
1,rap music ehh opp yeah know valid well know fa...,INTJ
2,preferably p hd low except wew lad video p min...,INTJ
3,drink like wish could drink red wine give head...,INTJ
4,space program ah bad deal meing freelance max ...,INTJ
...,...,...
106062,stay frustrate world life want take long nap w...,INFP
106063,fizzle around time mention sure mistake thing ...,INFP
106064,schedule modify hey w intp strong wing underst...,INFP
106065,enfj since january busy schedule able spend li...,INFP


In [41]:
def get_types2(row):
    t=row['type'] # or t

#    I = 0; N = 0
#    T = 0; J = 0

    if t[0] == 'I': I = 'I'
    elif t[0] == 'E': I = 'E'
    else: print('I-E not found')

    if t[1] == 'N': N = 'N'
    elif t[1] == 'S': N = 'S'
    else: print('N-S not found')

    if t[2] == 'T': T = 'T'
    elif t[2] == 'F': T = 'F'
    else: print('T-F not found')

    if t[3] == 'J': J = 'J'
    elif t[3] == 'P': J = 'P'
    else: print('J-P not found')
    return pd.Series( {'ie':I, 'ns':N , 'tf': T, 'jp': J })

